# A gallery of failures: what a vanilla PINN does to Chapter 6

**Book:** §5.1, Figure 5.1 &nbsp;·&nbsp; `ch05/vanilla_failures.ipynb`

Chapter 6 solves these four problems to three or four significant figures. It does so with hard
constraints, periodic embeddings, scaled residuals, and one network per material.

**This notebook strips all of that away** and solves the same four problems the naive way: soft
boundary penalties, plain inputs, one smooth network, the residual as written. All four fail.

None of them fails *loudly*. Every run below trains to a small, steadily decreasing loss. That is
the lesson of this chapter: **the loss going down is not evidence that the answer is right.**

| Panel | What breaks | Why | Fixed in |
|---|---|---|---|
| (a) Cylinder | collapses to $\psi\approx0$ | the trivial solution minimises the loss we wrote | §6.6 (hard BCs) |
| (b) Womersley | rel $L_2 \approx 6$ | residual terms differ by $O(25)$ | Appendix B (scaling) |
| (c) Composite wall | interface $T$ 3× wrong | a smooth net cannot have a kink | §5.3 (FBPINN) |
| (d) Stokes 2nd | the wave decays to nothing | time treated as just another coordinate | §5.5 (causality) |


In [ ]:
# Four of Chapter 6's problems, run with a VANILLA PINN.
# Soft boundary penalties, plain (x,t) inputs, one smooth network, the residual written
# exactly as the textbook writes it. Each cell below FAILS -- that is the point.
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt

def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s, seed=0):
    torch.manual_seed(seed); L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)

R = {}   # results

## (a) Potential flow past a cylinder — with *soft* boundary conditions

§6.6 solves this with a hard-constraint trial function. Here we do the obvious thing instead: penalise $\psi=0$ on the cylinder and $\psi\to Uy$ far away.

**Watch what happens.** $\psi\equiv 0$ satisfies Laplace's equation *exactly* and the cylinder condition *exactly*. It violates only one of three loss terms — so it is an excellent minimiser of the loss we wrote, and a useless answer to the question we asked. This is the **trivial-solution trap**.

In [ ]:
plt.figure(figsize=(7.5,4.6))
RO = 6.0
net = mlp([2,64,64,1], 0); opt = torch.optim.Adam(net.parameters(), 1e-3)
psi = lambda x, y: net(torch.cat([x/RO, y/RO], 1))
for e in range(4000):
    opt.zero_grad()
    r = 1 + (RO-1)*torch.rand(3000,1)**2; th = torch.rand(3000,1)*2*np.pi
    x = (r*torch.cos(th)).requires_grad_(True); y = (r*torch.sin(th)).requires_grad_(True)
    p = psi(x, y); res = g1(g1(p,x),x) + g1(g1(p,y),y)
    tb = torch.rand(400,1)*2*np.pi
    xb, yb = torch.cos(tb), torch.sin(tb)
    xo, yo = RO*torch.cos(tb), RO*torch.sin(tb)
    ((res**2).mean() + 10*(psi(xb,yb)**2).mean()
                     + 10*((psi(xo,yo) - yo)**2).mean()).backward(); opt.step()
thb = torch.linspace(0, 2*np.pi, 181).reshape(-1,1)
xc = torch.cos(thb).requires_grad_(True); yc = torch.sin(thb).requires_grad_(True)
p = psi(xc, yc); u = g1(p, yc); v = -g1(p, xc)
cp = (1-(u**2+v**2)).detach().numpy().ravel(); thn = thb.detach().numpy().ravel()
R['cyl_cpmin'] = float(cp.min())
plt.gca().plot(np.degrees(thn), 1-4*np.sin(thn)**2, 'g', lw=2.6, alpha=.6,
             label=r'exact $1-4\sin^2\theta$')
plt.gca().plot(np.degrees(thn), cp, 'r--', lw=1.8, label='vanilla PINN, soft BCs')
plt.gca().set_xlabel(r'$\theta$ (deg)'); plt.gca().set_ylabel('$C_p$')
plt.gca().grid(alpha=.3); plt.gca().legend(fontsize=9, loc='lower right')
plt.gca().set_title(f'(a) Cylinder (§6.6): the network finds $\\psi\\approx0$.\n'
                  f'$C_{{p,\\min}}={cp.min():+.2f}$, exact $-3.00$ — no flow at all',
                  fontsize=10.5)
print('(a) cyl Cp_min', R['cyl_cpmin'])
plt.tight_layout(); plt.show()

## (b) Womersley flow — with the residual left unscaled

§6.7 imposes no-slip and periodicity exactly, and scales the forcing so the velocity is $O(1)$. Here the residual is written as it stands: with $Wo=5$, $\omega=25$, the forcing term is $O(25)$ while the velocity is $O(1)$.

**The optimiser attends to the biggest term and ignores the rest.** The failure is one of *scaling*, not of physics — see Appendix B.

In [ ]:
plt.figure(figsize=(7.5,4.6))
NU, WO = 1.0, 5.0
W = WO**2; G = W; LAM = np.sqrt(1j*W/NU); TP = 2*np.pi/W
uex = lambda y, t: np.real((G/(1j*W))*(1 - np.cosh(LAM*y)/np.cosh(LAM))*np.exp(1j*W*t))
net = mlp([2,48,48,48,1], 4); opt = torch.optim.Adam(net.parameters(), 2e-3)
for e in range(6000):
    opt.zero_grad()
    y = (torch.rand(2000,1)*2-1).requires_grad_(True)
    t = (torch.rand(2000,1)*TP).requires_grad_(True)
    u = net(torch.cat([y,t],1))
    res = g1(u,t) - G*torch.cos(W*t) - NU*g1(g1(u,y),y)
    tb = torch.rand(300,1)*TP
    ((res**2).mean()
     + 10*(net(torch.cat([torch.full_like(tb,-1.), tb],1))**2).mean()
     + 10*(net(torch.cat([torch.full_like(tb, 1.), tb],1))**2).mean()).backward(); opt.step()
yg = np.linspace(-1,1,200); yt = torch.tensor(yg, dtype=torch.float32).reshape(-1,1)
errs = []
for ph, c in zip((0.0, 0.25, 0.5), ('tab:blue','tab:orange','tab:green')):
    tv = ph*TP
    with torch.no_grad(): up = net(torch.cat([yt, torch.full_like(yt, tv)],1)).numpy().ravel()
    ue = uex(yg, tv); errs.append(np.sqrt(np.mean((up-ue)**2)/np.mean(ue**2)))
    plt.gca().plot(ue, yg, c, lw=2.4, alpha=.5)
    plt.gca().plot(up, yg, '--', color=c, lw=1.5, label=f'phase {ph}T')
R['wom_err'] = float(np.mean(errs))
plt.gca().set_xlabel('u'); plt.gca().set_ylabel('y'); plt.gca().grid(alpha=.3)
plt.gca().legend(fontsize=8, loc='lower right')
plt.gca().set_title(f'(b) Womersley (§6.7): unscaled residual, soft BCs.\n'
                  f'rel $L_2$ = {R["wom_err"]:.2f} — solid exact, dashed PINN', fontsize=10.5)
print('(b) womersley err', R['wom_err'])
plt.tight_layout(); plt.show()

## (c) The composite wall — with one smooth network

§6.12 uses one network per layer, glued by flux continuity. Here we use a single $\tanh$ network across the whole wall.

**A $\tanh$ network is infinitely differentiable. It cannot have a kink.** So it returns the straight line through the endpoints, and the interface temperature comes out 3× wrong. No amount of extra training helps: the true solution is *not in the space the architecture can express*. The fix is architectural (§5.3, FBPINN).

In [ ]:
plt.figure(figsize=(7.5,4.6))
K1, K2, XI = 1.0, 5.0, 0.5
net = mlp([1,32,32,1], 2); opt = torch.optim.Adam(net.parameters(), 2e-3)
T_of = lambda x: net(x)
kfun = lambda x: torch.where(x < XI, torch.full_like(x, K1), torch.full_like(x, K2))
for e in range(4000):
    opt.zero_grad()
    x = torch.rand(1024,1).requires_grad_(True)
    res = g1(kfun(x)*g1(T_of(x), x), x)
    z, o = torch.zeros(1,1), torch.ones(1,1)
    ((res**2).mean() + 10*((T_of(z)-1)**2 + T_of(o)**2).sum()).backward(); opt.step()
xg = np.linspace(0,1,400); xt = torch.tensor(xg, dtype=torch.float32).reshape(-1,1)
with torch.no_grad(): Tp = T_of(xt).numpy().ravel()
R1, R2 = XI/K1, (1-XI)/K2; Tint = R2/(R1+R2)
Tex = np.where(xg < XI, 1-(1-Tint)*xg/XI, Tint*(1-xg)/(1-XI))
R['comp_pinn'] = float(np.interp(XI, xg, Tp)); R['comp_exact'] = float(Tint)
plt.gca().plot(xg, Tex, 'g', lw=2.6, alpha=.6, label=f'exact: kink, $T(\\xi)={Tint:.3f}$')
plt.gca().plot(xg, Tp, 'r--', lw=1.8, label=f'vanilla PINN: $T(\\xi)={R["comp_pinn"]:.3f}$')
plt.gca().axvline(XI, color='k', ls=':', lw=1)
plt.gca().text(XI+0.02, 0.85, '$k$ jumps $1\\to5$', fontsize=8.5)
plt.gca().set_xlabel('x'); plt.gca().set_ylabel('T'); plt.gca().grid(alpha=.3); plt.gca().legend(fontsize=9)
plt.gca().set_title('(c) Composite wall (§6.12): one smooth network.\n'
                  'A $\\tanh$ net cannot have a kink — so it invents none', fontsize=10.5)
print('(c) composite', R['comp_pinn'], 'vs', Tint)
plt.tight_layout(); plt.show()

## (d) Stokes' second problem — with time as an ordinary input

§6.5 makes time *exactly periodic* by feeding the network $(\sin\omega t, \cos\omega t)$ and needs no initial condition at all. Here we feed it $t$ and start from rest, over ten forcing periods.

**The wave dies.** The network finds that $u\approx 0$ nearly solves the PDE over most of the domain, and has no notion that the wall's motion must *propagate forward in time*. This is **causality violation** (§5.5).

In [ ]:
plt.figure(figsize=(7.5,4.6))
NU, W, Y = 1.0, 2*np.pi, 4.0
NPER = 10; TMAX = NPER*2*np.pi/W; kk = np.sqrt(W/(2*NU))
net = mlp([2,48,48,48,1], 3); opt = torch.optim.Adam(net.parameters(), 2e-3)
for e in range(6000):
    opt.zero_grad()
    y = (torch.rand(2000,1)*Y).requires_grad_(True)
    t = (torch.rand(2000,1)*TMAX).requires_grad_(True)
    u = net(torch.cat([y,t],1)); res = g1(u,t) - NU*g1(g1(u,y),y)
    tb = torch.rand(300,1)*TMAX; yb = torch.rand(300,1)*Y
    ((res**2).mean()
     + 10*((net(torch.cat([torch.zeros_like(tb), tb],1)) - torch.cos(W*tb))**2).mean()
     + 10*(net(torch.cat([torch.full_like(tb, Y), tb],1))**2).mean()
     + 10*(net(torch.cat([yb, torch.zeros_like(yb)],1))**2).mean()).backward(); opt.step()
yg = np.linspace(0, Y, 200); yt = torch.tensor(yg, dtype=torch.float32).reshape(-1,1)
tp = TMAX - 0.25*(2*np.pi/W)
with torch.no_grad(): up = net(torch.cat([yt, torch.full_like(yt, tp)],1)).numpy().ravel()
ue = np.exp(-kk*yg)*np.cos(W*tp - kk*yg)
R['stokes2_err'] = float(np.sqrt(np.mean((up-ue)**2)/np.mean(ue**2)))
R['stokes2_peak'] = float(np.abs(up).max())
plt.gca().plot(ue, yg, 'g', lw=2.6, alpha=.6, label='exact Stokes layer')
plt.gca().plot(up, yg, 'r--', lw=1.8, label=f'vanilla PINN (rel $L_2$={R["stokes2_err"]:.2f})')
plt.gca().plot(np.exp(-kk*yg), yg, 'k:', lw=1, alpha=.5, label=r'$e^{-ky}$ envelope')
plt.gca().plot(-np.exp(-kk*yg), yg, 'k:', lw=1, alpha=.5)
plt.gca().set_xlabel('u'); plt.gca().set_ylabel('y'); plt.gca().grid(alpha=.3)
plt.gca().legend(fontsize=8.5, loc='lower right')
plt.gca().set_title(f'(d) Stokes 2nd (§6.5): naive time, 10 periods.\n'
                  f'The wave dies: peak $|u|$ = {R["stokes2_peak"]:.3f}, not $1$', fontsize=10.5)
print('(d) stokes2', R['stokes2_err'], R['stokes2_peak'])


plt.tight_layout(); plt.show()

## The moral

Four reasonable networks. Four reasonable optimisers. Four losses that went down.

And four wrong answers — because what the optimiser minimised was not what we wanted: the trivial solution was admissible (a), the terms were incommensurable (b), the answer was outside the network's reach (c), and time was just another coordinate (d).

**A PINN is only as good as the problem you actually posed to it.** The rest of Chapter 5 is about learning to tell the difference.

In [ ]:
for k, v in R.items():
    print(f'{k:16s} {v}')